# ovarian-agent-contract: LLM extraction on Colab

Runs the pinned open-weight model (vLLM, JSON-schema constrained decoding, token logprobs) on the **dev and
calibration sites only**. Test sites stay locked in code until `eval/PROTOCOL.md` is frozen.

Before you start:
1. On your laptop: `uv run python -m oac.bundle`, then upload `dist/oac-code.zip` to Google Drive at `MyDrive/oac/`.
2. Runtime > Change runtime type > GPU (T4 is enough; L4/A100 just run faster; the model is the same on all).

Outputs go to `MyDrive/oac/runs/`. They contain report quotes, so keep them out of git (the repo ignores `runs/`).
Every cell is safe to re-run: the runner skips reports that are already done.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 1. Code (from the bundle on Drive)

In [ ]:
import os, shutil, zipfile
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/oac"
CODE = "/content/oac"
if os.path.exists(CODE):
    shutil.rmtree(CODE)
zipfile.ZipFile(f"{WORK}/oac-code.zip").extractall(CODE)
print("code commit:", open(f"{CODE}/COMMIT").read().strip())

## 2. Install

vLLM brings its own PyTorch build; this takes a few minutes. After the first successful run, write the installed
version into `VLLM_SPEC` (for example `"vllm==0.34.0"`) so later runs use exactly the same engine.

In [ ]:
VLLM_SPEC = "vllm"
!pip -q install "{VLLM_SPEC}"
!pip -q install -e {CODE}
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 3. Public data (sha256-checked)

In [ ]:
os.environ["OAC_DATA_DIR"] = "/content/data"
!cd {CODE} && python -m oac.fetch

## 4. Pinned model

Qwen3-8B-AWQ (Apache-2.0), fixed to one Hugging Face commit. It fits a 16 GB T4 in float16, so results do not
depend on which GPU Colab assigns. Thinking mode is off (`enable_thinking=False`).

In [ ]:
MODEL = "Qwen/Qwen3-8B-AWQ"
REVISION = "4da05a8edb55c6046cce958586c33b61da07bb79"
RUNS = f"{WORK}/runs"
COMMON = f"--backend vllm --model {MODEL} --revision {REVISION} --dtype half --max-model-len 8192"

## 5. Smoke test on 3 synthetic reports

If this fails, copy the last 30 lines of the error back to the chat. Common T4 issues: out of memory
(lower `--gpu-mem` to 0.85 or `--max-model-len` to 6144) or an attention backend that does not support the GPU.

In [ ]:
!cd {CODE} && rm -rf /content/smoke && python -m oac.run_llm {COMMON} --synthetic --k 2 --out /content/smoke
import json
for line in open("/content/smoke/outputs.jsonl"):
    o = json.loads(line)
    print(o["case_id"], {f["field"]: (f["status"], f["value"]) for f in o["fields"]})

## 6. System run: hybrid extraction, greedy + 5 samples (dev + calibration sites, 184 reports)

Run time on a T4 is not measured yet (the manifest records start and finish times). Re-run the cell if Colab disconnects; finished reports are skipped.

In [ ]:
!cd {CODE} && python -m oac.run_llm {COMMON} --splits dev,calibration --mode hybrid --k 5 --out {RUNS}/qwen3-8b-awq_hybrid_k5

## 7. Baseline 2: end-to-end (the model also stages directly), greedy only

In [ ]:
!cd {CODE} && python -m oac.run_llm {COMMON} --splits dev,calibration --mode e2e --k 0 --out {RUNS}/qwen3-8b-awq_e2e

## 8. Summary (statuses only, no report text) and download

Download `runs.zip` from Drive and unzip it into the repo's `runs/` folder.

In [ ]:
import collections, glob
for p in sorted(glob.glob(f"{RUNS}/*/outputs.jsonl")):
    c = collections.Counter()
    for line in open(p):
        for f in json.loads(line)["fields"]:
            c[f["status"]] += 1
    print(p.split("/")[-2], sum(1 for _ in open(p)), "reports", dict(c))
shutil.make_archive(f"{WORK}/runs", "zip", RUNS)
print("wrote", f"{WORK}/runs.zip")